In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

use utils to define naming for __bronze_, silver and gold databases_

'%run' is used to run a notebook

In [0]:
%run /Workspace/Users/hamsaavarshinib11@gmail.com/consolidated_pipeline/setup/utiliies

In [0]:
print(bronze_schema)

## Bronze schema

In [0]:
# can be used to specify ENVIRONMENT, different CATALOGS and BUCKETS
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog_value = dbutils.widgets.get("catalog")
data_source_value = dbutils.widgets.get("data_source")
print(catalog_value,data_source_value)
#specify the buckets
base_path = f's3://sportsbar-dp-hamz/{data_source_value}/*.csv'
print(base_path)

In [0]:
#create a dataframe
df = spark.read.format("csv").load(base_path)
display(df.limit(10))

In [0]:
df.show(5)

In [0]:
df = (
    spark.read.format("csv")
        .option("header",True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
# here in the schema now, the customer_id is integer,  since its bronze layer ie. raw data, no need to change it to string, but we will do it later

In [0]:
#now save the data in bronze schema
df.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", True) \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_value}.{bronze_schema}.{data_source_value}")
#CDF enableChangeDataFeed --> allows to track the changes at row level (audit all the changes even for row of data) for time travel

## Silver processing

silver will contain clean data (remove duplicates, null values, leading spaces)

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog_value}.{bronze_schema}.{data_source_value};")
display(df_bronze.limit(10 ))

In [0]:
df_bronze.printSchema()

TRANSFORMATION

1. Drop Duplicates

In [0]:
#find duplicate customer_ids
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count")>1)
display(df_duplicates)

In [0]:
print("rows before duplicates removed: ", df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("rows after duplicates removed: ", df_silver.count())

2. Trim spaces in customer name

In [0]:
#check for values with extra spaces
display(df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name"))))

In [0]:
#remove those spaces
df_silver = df_silver.withColumn("customer_name", F.trim(F.col("customer_name")))

In [0]:
# # Sanity Check

# # check those values
# display(
#     df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
# )

3. Data Quality Fix: correct city typos

In [0]:
df_silver.select('city').distinct().show()

In [0]:
# # typo dictionary
# city_typos = {
#     'Bengaluru': ['Bengaluruu', 'Bengaluruu', 'Bengalore'],
#     'Hyderabad': ['Hyderabadd', 'Hyderbad'],
#     'New Delhi': ['NewDelhi', 'NewDheli', 'NewDelhee']
# }

# typos → correct names
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)

In [0]:
#sanity check
df_silver.select('city').distinct().show()

4. Fix title case issue

In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
# Title case fix
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name"))
)
#initcap --> 1st letter capital and others are small

In [0]:
# sanity check
df_silver.select('customer_name').distinct().show()

5. handle missing values in city

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:

# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

#create a data frame and do a join to assign the cities for the null ones
df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")   # Replace null with fixed city
    )
    .drop("fixed_city")
)

In [0]:
# Sanity Checks
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

6. convert customer_id to string because the gold layer has customer_id as string

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

### Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
# the columns customer_name and customer_id are different in silver and gold, and gold has three extra columns - market, platform, and channel
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    # use of coalesce - if the city is null, it appends Unknown
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

In [0]:
display(df_silver.limit(5))

In [0]:
#write the changes into silver layer
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog_value}.{silver_schema}.{data_source_value}")

# Gold

In [0]:
#save the processed silver layer into gold layer
df_silver = spark.sql(f"SELECT * FROM {catalog_value}.{silver_schema}.{data_source_value};")


# take req cols only
# "customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog_value}.{gold_schema}.sb_dim_{data_source_value}")

### ## Merging data source with parent

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)
# in child table, rename with customer_id to cusomter_code since that is the name in parent table

In [0]:
#UPSERT OPERATION (update if matched, insert if not matched)
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()